## Crear descarga reproducible

La descarga se centraliza en `src/inf8239_u01/data.py` mediante la función
`download_csv(url)`, de modo que cualquier persona pueda reconstruir el dataset
ejecutando el notebook, sin rutas personales.

La fuente aprobada (*Air Quality and Pollution Assessment*, Kaggle · Apache 2.0)
**requiere autenticación**, por lo que no expone una URL `.csv` directa. Tal como
indica la guía para fuentes con API/autenticación, ese procedimiento se
**encapsula** dentro de `download_csv`:

- Si la URL es una página de dataset de Kaggle, se extrae el *slug*
  (`owner/dataset`) y se descarga con **`kagglehub`** (API autenticada).
- El CSV se copia a la ruta **estable** `data/raw/dataset.csv`, resuelta contra la
  raíz del proyecto — nunca se usa `C:\Users\...` ni `/content/drive/...`.
- Solo se permiten los datasets **aprobados por el docente** (validación por *slug*).

**Requisitos previos (una sola vez):** `pip install kagglehub` y credenciales de
Kaggle en `~/.kaggle/kaggle.json` (o las variables `KAGGLE_USERNAME` /
`KAGGLE_KEY`). Guía oficial: https://www.kaggle.com/docs/api

In [2]:
import sys
sys.path.append("..")  # raíz del proyecto para poder importar src/

from src.inf8239_u01.data import download_csv

URL = "https://www.kaggle.com/datasets/mujtabamatin/air-quality-and-pollution-assessment"
path = download_csv(URL)
print(path)

c:\Users\Mudstart\Documents\Documentos\Maestria en Ciencia de Datos e Inteligencia Artificial\Ciencia de Datos II\INF8239_U01\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\Mudstart\Documents\Documentos\Maestria en Ciencia de Datos e Inteligencia Artificial\Ciencia de Datos II\INF8239_U01\data\raw\dataset.csv


### Comentario de la salida

`download_csv` imprime la ruta local del archivo descargado
(`data/raw/dataset.csv`). Que exista un único CSV en esa ruta confirma que la
descarga reproducible funcionó. El nombre del archivo se estandariza a
`dataset.csv` a propósito, para que el resto del pipeline lea siempre la misma ruta
independientemente del nombre original en Kaggle.

## Cargar y reconocer el esquema

Se carga el CSV descargado en el paso anterior y se realiza una primera inspección
de su estructura, antes de cualquier limpieza o modelado.

- **`pd.read_csv("../data/raw/dataset.csv")`** — lee el archivo desde la ruta estable
  del proyecto (`../` porque el notebook vive en `notebooks/`).
- **`df.shape`** — dimensiones (filas × columnas).
- **`df.dtypes`** — tipo de dato de cada columna, para distinguir variables numéricas
  de categóricas.
- **`df.head()` / `df.tail()`** — primeras y últimas filas, para confirmar que los
  datos se cargaron correctamente y detectar anomalías obvias.
- **`assert not df.empty`** — verificación mínima de que el dataset no está vacío.

**Resultado esperado:** 5,000 filas y 10 columnas (9 predictores + el target
`Air Quality`).

In [3]:
import pandas as pd
df = pd.read_csv("../data/raw/dataset.csv")
print(df.shape)
print(df.dtypes)
print(df.head())
print(df.tail())
assert not df.empty

(5000, 10)
Temperature                      float64
Humidity                         float64
PM2.5                            float64
PM10                             float64
NO2                              float64
SO2                              float64
CO                               float64
Proximity_to_Industrial_Areas    float64
Population_Density                 int64
Air Quality                          str
dtype: object
   Temperature  Humidity  PM2.5  PM10   NO2   SO2    CO  \
0         29.8      59.1    5.2  17.9  18.9   9.2  1.72   
1         28.3      75.6    2.3  12.2  30.8   9.7  1.64   
2         23.1      74.7   26.7  33.8  24.4  12.6  1.63   
3         27.1      39.1    6.1   6.3  13.5   5.3  1.15   
4         26.5      70.7    6.9  16.0  21.9   5.6  1.01   

   Proximity_to_Industrial_Areas  Population_Density Air Quality  
0                            6.3                 319    Moderate  
1                            6.0                 611    Moderate  
2        

### Comentario de la salida

La carga confirma la estructura esperada del dataset:

- **Dimensiones:** 5,000 filas y 10 columnas.
- **Predictores (9), todos numéricos:** temperatura, humedad, PM2.5, PM10, NO2, SO2,
  CO, proximidad a zonas industriales y densidad poblacional (`float64` / `int64`).
- **Target (`Air Quality`):** de tipo texto (`object`), con las categorías Good,
  Moderate, Poor y Hazardous.

Que las 9 variables predictoras sean numéricas confirma lo previsto: el dataset es
directamente apto para `StandardScaler` + SVM. El target es categórico (texto), lo
que define la tarea como **clasificación multiclase** y será la columna a separar en
el paso 7.

> ℹ️ Confirma el nombre exacto de la columna target en la salida de `df.dtypes` (por
> ejemplo `Air Quality`), ya que se usará como `TARGET` más adelante.

## Construir la auditoría

Se construye una tabla de auditoría que resume el estado de cada columna antes de
modelar, para tomar decisiones con evidencia y no de forma automática.

Por cada variable, `audit` reporta:

- **`tipo`** — tipo de dato de la columna.
- **`ausentes`** — cantidad de valores faltantes (NaN).
- **`porcentaje_ausente`** — proporción de faltantes (%), ordenada de mayor a menor.
- **`unicos`** — número de valores distintos (incluyendo NaN).

Además se cuentan los **duplicados** de filas. Esta radiografía permite decidir, con
justificación, si hará falta imputar, transformar o conservar cada columna.

In [4]:
audit = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "ausentes": df.isna().sum(),
    "porcentaje_ausente": (df.isna().mean()*100).round(2),
    "unicos": df.nunique(dropna=False)
}).sort_values("porcentaje_ausente", ascending=False)
print("Duplicados:", df.duplicated().sum())
display(audit)

Duplicados: 0


,tipo,ausentes,porcentaje_ausente,unicos
Temperature,float64,0,0.0,362
Humidity,float64,0,0.0,723
PM2.5,float64,0,0.0,815
PM10,float64,0,0.0,955
NO2,float64,0,0.0,445
SO2,float64,0,0.0,348
CO,float64,0,0.0,265
Proximity_to_Industrial_Areas,float64,0,0.0,179
Population_Density,int64,0,0.0,683
Air Quality,str,0,0.0,4


### Comentario de la salida

La auditoría muestra un dataset limpio y listo para modelar:

- **Duplicados: 0** — no hay filas repetidas.
- **`porcentaje_ausente`: 0.00 en todas las columnas** — no hay valores faltantes
  que imputar (dataset sintético bien formado).
- **Tipos:** las 9 variables predictoras son numéricas (`float64` / `int64`) y el
  target `Air Quality` es categórico (`object`).
- **`unicos`:** los predictores presentan muchos valores distintos (coherente con
  mediciones continuas), mientras que el target tiene solo **4 valores** (Good,
  Moderate, Poor, Hazardous), confirmando la naturaleza multiclase del problema.

Como no hay duplicados, ni valores faltantes, ni columnas no numéricas entre los
predictores, **no existe justificación para eliminar filas o columnas** en esta
etapa: se preservan íntegros los datos.

> ℹ️ Si algún valor difiere de lo anterior (p. ej. aparecen faltantes), documéntalo y
> ajusta la estrategia de imputación en el preprocesamiento del paso 8.


### Diccionario de datos

| Variable | Significado | Unidad | Fuente | Disponible al predecir | Transformación prevista | Riesgo |
|----------|-------------|--------|--------|------------------------|--------------------------|--------|
| Temperature | Temperatura media de la región | °C | Medición ambiental | Sí | StandardScaler | Posibles outliers estacionales |
| Humidity | Humedad relativa | % | Medición ambiental | Sí | StandardScaler | Valores fuera de rango [0–100] |
| PM2.5 | Partículas finas | µg/m³ | Sensor de calidad del aire | Sí | StandardScaler | Correlación con PM10 |
| PM10 | Partículas gruesas | µg/m³ | Sensor de calidad del aire | Sí | StandardScaler | Correlación con PM2.5 |
| NO2 | Dióxido de nitrógeno | ppb | Sensor de calidad del aire | Sí | StandardScaler | Outliers en zonas industriales |
| SO2 | Dióxido de azufre | ppb | Sensor de calidad del aire | Sí | StandardScaler | Outliers en zonas industriales |
| CO | Monóxido de carbono | ppm | Sensor de calidad del aire | Sí | StandardScaler | Escala distinta al resto |
| Proximity_to_Industrial_Areas | Distancia a la zona industrial más cercana | km | Dato geográfico | Sí | StandardScaler | Proxy de contaminación |
| Population_Density | Densidad poblacional | hab/km² | Dato demográfico | Sí | StandardScaler | Sesgo urbano/rural |
| Air Quality | Nivel de calidad del aire (**target**) | categoría | Etiqueta derivada | — (es la salida) | Etiqueta de clasificación | Clases posiblemente desbalanceadas |

## Definir target y retirar fugas

Se separa la variable objetivo (`y`) del conjunto de predictores (`X`) y se revisan
posibles fugas de información (*data leakage*).

- **`TARGET = "Air Quality"`** — la columna a predecir (nivel de calidad del aire).
- **`DROP_COLUMNS = []`** — no se elimina ninguna columna. A diferencia de otros
  datasets (que traen identificadores como `customer_id`), aquí las **9 variables son
  mediciones ambientales legítimas**, disponibles al momento de predecir y sin
  relación directa con la etiqueta. No hay identificadores ni fugas que retirar.
- Los `assert` verifican que el target exista, no tenga nulos y tenga al menos 2
  clases.

Se respeta el criterio de la guía: **no se elimina ninguna columna solo porque
empeore la métrica**; como ninguna representa fuga ni identificador, todas se
conservan por su valor semántico.

In [5]:
TARGET = "Air Quality"      # confirma el nombre exacto con df.columns.tolist()
DROP_COLUMNS = []           # no hay identificadores ni fugas: las 9 son features válidas
assert TARGET in df.columns
X = df.drop(columns=[TARGET] + DROP_COLUMNS)
y = df[TARGET]
print(y.value_counts(dropna=False))
assert y.notna().all()
assert y.nunique() >= 2

Air Quality
Good         2000
Moderate     1500
Poor         1000
Hazardous     500
Name: count, dtype: int64


### Comentario de la salida

`y.value_counts()` muestra la distribución de las 4 clases del target `Air Quality`
(Good, Moderate, Poor, Hazardous). Los `assert` pasan sin error, confirmando que:

- El target existe en el DataFrame.
- No tiene valores nulos (`y.notna().all()`).
- Tiene al menos 2 clases (`y.nunique() >= 2`) — en este caso, 4.

**Observa el balance entre clases:** si las cuatro categorías tienen conteos
similares, el problema está equilibrado; si alguna (p. ej. "Hazardous") es
minoritaria, se justifica aún más usar **F1 macro** como métrica y estratificar la
partición, para que el modelo no ignore las clases menos frecuentes —que suelen ser
las de mayor interés (aire peligroso).

Con `X` (9 predictores) e `y` (target) ya separados, los datos quedan listos para
definir el preprocesamiento en el paso 8.

## Separar tipos y crear preprocesamiento

Se identifican automáticamente las columnas numéricas y categóricas, y se define un
preprocesamiento encapsulado en un **`ColumnTransformer`**, que aplica el tratamiento
adecuado a cada tipo dentro de un pipeline (evitando *data leakage*).

- **`num_cols` / `cat_cols`** — separan las columnas por tipo con `select_dtypes`.
- **`num_pipe`** — para variables numéricas: imputa faltantes con la **mediana** y
  luego estandariza con **`StandardScaler`** (imprescindible para SVM).
- **`cat_pipe`** — para variables categóricas: imputa con la **moda** y codifica con
  **`OneHotEncoder(handle_unknown="ignore")`**.
- **`preprocess`** — combina ambos pipelines por columnas.

En este dataset **las 9 predictoras son numéricas**, así que solo actúa la rama
numérica. La rama categórica se deja definida por robustez y generalidad, pero no
recibe columnas.

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()
num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),
                     ("scale", StandardScaler())])
cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                     ("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocess = ColumnTransformer([("num", num_pipe, num_cols),
                                ("cat", cat_pipe, cat_cols)])
print(len(num_cols), len(cat_cols))

9 0


### Comentario de la salida

La salida `9 0` confirma lo esperado:

- **9 columnas numéricas** — todas las predictoras (temperatura, humedad, PM2.5,
  PM10, NO2, SO2, CO, proximidad industrial y densidad poblacional).
- **0 columnas categóricas** — no hay predictores de texto que codificar.

Por tanto, el preprocesamiento se reduce en la práctica a **imputación por mediana +
estandarización** sobre las 9 variables. Al estar todo dentro del `ColumnTransformer`
(y este dentro del pipeline en el paso 9), el escalador se ajustará **solo con los
datos de entrenamiento** de cada partición, respetando la separación train/test y
evitando fuga de información.

## Dividir, baseline y SVM

Se separan los datos en entrenamiento y prueba, se entrena una línea base y un modelo
SVM, y se comparan sobre el conjunto de prueba.

- **`train_test_split(..., test_size=.20, random_state=42, stratify=y)`** — reserva el
  20% para prueba de forma **reproducible** y **estratificada**, manteniendo la
  proporción de las 4 clases en ambos conjuntos.
- **`dummy`** — línea base (`DummyClassifier`, estrategia "most_frequent"): predice
  siempre la clase mayoritaria. Es el mínimo a superar.
- **`svm`** — pipeline `preprocess` + `SVC(kernel rbf por defecto, C=1, gamma="scale")`.
  El preprocesamiento va **dentro** del pipeline, así el escalado se ajusta solo con
  los datos de entrenamiento.
- Se evalúan ambos con **F1 macro** (da igual peso a las 4 clases, apropiado si hay
  desbalance) y se imprime el `classification_report` del SVM.

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, f1_score

Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.20,random_state=42,stratify=y)
dummy=Pipeline([("prep",preprocess),("model",DummyClassifier(strategy="most_frequent"))])
svm=Pipeline([("prep",preprocess),("model",SVC(C=1,gamma="scale",probability=True,random_state=42))])
for name,model in {"dummy":dummy,"svm":svm}.items():
    model.fit(Xtr,ytr)
    pred=model.predict(Xte)
    print(name, f1_score(yte,pred,average="macro"))
print(classification_report(yte,svm.predict(Xte)))

dummy 0.14285714285714285


c:\Users\Mudstart\Documents\Documentos\Maestria en Ciencia de Datos e Inteligencia Artificial\Ciencia de Datos II\INF8239_U01\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


svm 0.9126733637152503
              precision    recall  f1-score   support

        Good       0.99      0.99      0.99       400
   Hazardous       0.88      0.82      0.85       100
    Moderate       0.94      0.96      0.95       300
        Poor       0.86      0.86      0.86       200

    accuracy                           0.94      1000
   macro avg       0.92      0.91      0.91      1000
weighted avg       0.94      0.94      0.94      1000



### Comentario de la salida

Comparación de los dos modelos sobre el conjunto de prueba (F1 macro):

- **`dummy`:** valor bajo. Al predecir siempre la clase mayoritaria, falla por
  completo en las otras 3 clases, por lo que su F1 macro es pobre. Es la vara mínima.
- **`svm`:** valor claramente superior al dummy, lo que confirma que el modelo sí
  aprende patrones útiles a partir de las variables ambientales.

En el `classification_report` del SVM, revisa **por clase** (Good, Moderate, Poor,
Hazardous):

- La **precision** y el **recall** de cada nivel de calidad del aire.
- Presta especial atención al **recall de las clases de mayor riesgo** (Poor /
  Hazardous): es el que interesa maximizar, porque el error más costoso es no detectar
  aire peligroso (definido en la ficha del paso 1).
- El **F1 macro** resume el desempeño equilibrado entre las 4 clases.

> ✍️ Anota aquí tus valores reales (F1 macro del dummy y del SVM) tras ejecutar la
> celda, para dejar constancia de la mejora del SVM sobre la línea base.